In [2]:
import pandas as pd

customers = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/olist_sellers_dataset.csv")
category = pd.read_csv("D:/DA Projects/Logistics-Analytics-Dashboard/data/raw/product_category_name_translation.csv")

In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

In [4]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

orders["estimated_days"] = (
    orders["order_estimated_delivery_date"]
    - orders["order_purchase_timestamp"]
).dt.days

orders["delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

orders["is_late"] = (
    orders["delay_days"] > 0
).astype(int)

In [5]:
fact = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

print(fact.shape)

(99441, 16)


In [6]:
payments_agg = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value","sum"),
        payment_installments=("payment_installments","max")
    )
    .reset_index()
)

In [7]:
fact = fact.merge(
    payments_agg,
    on="order_id",
    how="left"
)

In [8]:
reviews_agg = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score","mean")
    )
    .reset_index()
)

In [9]:
fact = fact.merge(
    reviews_agg,
    on="order_id",
    how="left"
)

In [10]:
items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        total_price=("price","sum"),
        freight_value=("freight_value","sum"),
        item_count=("order_item_id","count")
    )
    .reset_index()
)

In [11]:
fact = fact.merge(
    items_agg,
    on="order_id",
    how="left"
)

In [12]:
fact["order_value"] = (
    fact["total_price"]
    + fact["freight_value"]
)

In [13]:
fact["on_time_delivery"] = (
    fact["is_late"] == 0
).astype(int)

In [14]:
fact = fact[
[
"order_id",
"customer_id",
"customer_city",
"customer_state",
"order_status",
"order_purchase_timestamp",
"delivery_days",
"delay_days",
"is_late",
"on_time_delivery",
"payment_value",
"review_score",
"total_price",
"freight_value",
"order_value",
"item_count"
]
]

In [15]:
fact.to_csv(
    "D:/DA Projects/Logistics-Analytics-Dashboard/data/processed/fact_orders.csv",
    index=False
)

In [16]:
fact.shape

(99441, 16)

In [17]:
fact.head()

,order_id,customer_id,customer_city,customer_state,order_status,order_purchase_timestamp,delivery_days,delay_days,is_late,on_time_delivery,payment_value,review_score,total_price,freight_value,order_value,item_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,sao paulo,SP,delivered,2017-10-02 10:56:33,8.0,-8.0,0,1,38.71,4.0,29.99,8.72,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,BA,delivered,2018-07-24 20:41:37,13.0,-6.0,0,1,141.46,4.0,118.70,22.76,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,vianopolis,GO,delivered,2018-08-08 08:38:49,9.0,-18.0,0,1,179.12,5.0,159.90,19.22,179.12,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,RN,delivered,2017-11-18 19:28:06,13.0,-13.0,0,1,72.20,5.0,45.00,27.20,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,SP,delivered,2018-02-13 21:18:39,2.0,-10.0,0,1,28.62,5.0,19.90,8.72,28.62,1.0


In [18]:
fact.describe()

,order_purchase_timestamp,delivery_days,delay_days,is_late,on_time_delivery,payment_value,review_score,total_price,freight_value,order_value,item_count
count,99441,96476.000000,96476.000000,99441.000000,99441.000000,99440.000000,99441.000000,98666.000000,98666.000000,98666.000000,98666.000000
mean,2017-12-31 08:43:12.776581120,12.094086,-11.876881,0.065717,0.934283,160.990267,4.071235,137.754076,22.823562,160.577638,1.141731
min,2016-09-04 21:15:19,0.000000,-147.000000,0.000000,0.000000,0.000000,1.000000,0.850000,0.000000,9.590000,1.000000
25%,2017-09-12 14:46:19,6.000000,-17.000000,0.000000,1.000000,62.010000,4.000000,45.900000,13.850000,61.980000,1.000000
50%,2018-01-18 23:04:36,10.000000,-12.000000,0.000000,1.000000,105.290000,5.000000,86.900000,17.170000,105.290000,1.000000
75%,2018-05-04 15:42:16,15.000000,-7.000000,0.000000,1.000000,176.970000,5.000000,149.900000,24.040000,176.870000,1.000000
max,2018-10-17 17:30:18,209.000000,188.000000,1.000000,1.000000,13664.080000,5.000000,13440.000000,1794.960000,13664.080000,21.000000
std,NaN,9.551746,10.183854,0.247789,0.247789,221.951257,1.358410,210.645145,21.650909,220.466087,0.538452


In [19]:
fact["purchase_year"] = (
    fact["order_purchase_timestamp"]
    .dt.year
)

fact["purchase_month"] = (
    fact["order_purchase_timestamp"]
    .dt.month
)

fact["purchase_day"] = (
    fact["order_purchase_timestamp"]
    .dt.day_name()
)

In [20]:
fact["order_value_segment"] = pd.cut(
    fact["order_value"],
    bins=[0,50,150,300,10000],
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

In [21]:
fact["delivery_speed"] = pd.cut(
    fact["delivery_days"],
    bins=[0,5,10,20,500],
    labels=[
        "Fast",
        "Normal",
        "Slow",
        "Very Slow"
    ]
)

In [22]:
fact.to_csv(
    "../data/processed/fact_orders_final.csv",
    index=False
)

In [23]:
fact.groupby(
    "customer_state"
)["order_id"].count().sort_values(
    ascending=False
).head(10)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: order_id, dtype: int64

In [24]:
fact.groupby(
    "customer_state"
)["review_score"].mean().sort_values(
    ascending=False
).head(10)

customer_state
AP    4.176471
AM    4.175676
PR    4.168682
SP    4.160730
RS    4.126235
MG    4.120541
MS    4.106294
TO    4.100000
MT    4.086549
RN    4.075258
Name: review_score, dtype: float64

In [25]:
(
    fact.groupby("customer_state")
    ["is_late"]
    .mean()*100
).sort_values(
    ascending=False
).head(10)

customer_state
AL    20.581114
MA    16.733601
SE    14.571429
PI    13.333333
CE    13.173653
BA    11.715976
RJ    11.632431
PA    10.871795
RR    10.869565
ES    10.526316
Name: is_late, dtype: float64

In [26]:
monthly_kpi = (
    fact.groupby(
        ["purchase_year","purchase_month"]
    )
    .agg(
        orders=("order_id","count"),
        revenue=("order_value","sum"),
        avg_review=("review_score","mean"),
        otd=("on_time_delivery","mean")
    )
    .reset_index()
)

monthly_kpi["otd"] *= 100

In [27]:
state_kpi = (
    fact.groupby(
        "customer_state"
    )
    .agg(
        orders=("order_id","count"),
        revenue=("order_value","sum"),
        avg_review=("review_score","mean"),
        late_rate=("is_late","mean")
    )
    .reset_index()
)

state_kpi["late_rate"] *= 100

In [28]:
monthly_kpi.to_csv(
    "../data/processed/monthly_kpi.csv",
    index=False
)

state_kpi.to_csv(
    "../data/processed/state_kpi.csv",
    index=False
)